In [1]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings , ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


C:\Users\Admin\AppData\Local\Temp\ipykernel_9176\1244829720.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
data_dir = Path("data")
faiss_dir = Path("faiss_index")

In [3]:
embed_model = "nomic-embed-text"
chat_model = "llama3.2"

In [4]:
#load pdfs & split into chunks
pdf_files = sorted(data_dir.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs:", [p.name for p in pdf_files])

all_docs = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    all_docs.extend(loader.load())

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 900, 
    chunk_overlap = 150,
)
chunks = splitter.split_documents(all_docs)

print(f"Loaded {len(all_docs)} pages -> split into {len(chunks)} chunks")
print("\nSample chunk preview:\n", chunks[0].page_content[:400])

Found 4 PDFs: ['CR_TTO_28032025.pdf', 'ir-pack-tnc-19042023.pdf', 'tnc.pdf', 'TRAI_Handbook_2018_Eng.pdf']


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 36 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 68 0 (offset 0)
Ignoring wrong pointing object 93 0 (offset 0)
Ignoring wrong pointing object 95 0 (offset 0)
Ignoring wrong pointing object 97 0 (offset 0)
Ignoring wrong pointing object 100 0 (offset 0)
Ignoring wrong pointing object 102 0 (offset 0)
Ignoring wrong pointing object 105 0 (offset 0)
Ignoring wrong pointing object 107 0 (offset 0)
Ignoring wrong pointing object 109 0 (offset 0)
Ignoring wrong pointing object 111 0 (offset 0)
Ignoring wrong pointing object 122 0 (offset 0)
Ignoring wrong pointing object 124 0 (offset 0)
Ignoring wrong pointing object 127 0 (offset 0)
Ignoring wrong pointing object 129 0 (offset 0)
Ignor

Loaded 121 pages -> split into 391 chunks

Sample chunk preview:
 1  
 
 
 
TELECOM REGULATORY AUTHORITY OF INDIA 
 
 
 
 
 
CONSOLIDATED TELECOMMUNICATION TARIFF ORDER (TTO), 2025 
(UPDATED TILL 70TH AMENDMENT) 
 
 
No. ---- of 2025 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
DISCLAIMER 
 
This compendium is intended for informational purposes only. While preparing this 
compendium, every effort has been made to ensure the accuracy and completeness


In [9]:
# build and save FAISS vector store 
embeddings = OllamaEmbeddings(model=embed_model)

if faiss_dir.exists():
    vectorstore = FAISS.load_local(
        str(faiss_dir),
        embeddings, 
        allow_dangerous_deserialization=True,
    )
    print("loaded existing FAISS index")

else:
    vectorstore = FAISS.from_documents(chunks, embeddings)
    faiss_dir.mkdir(exist_ok= True)
    vectorstore.save_local(str(faiss_dir))
    print("Built and saved new faiss index")

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

loaded existing FAISS index


In [10]:
# define prompt & rag chain
def format_docs_with_sources(docs):
    formatted = []
    for i , doc in enumerate(docs, start=1):
        source = Path(doc.metadata.get("source", "unknown")).name
        page = doc.metadata.get("page", 0)+ 1
        formatted.append(f"[Source {i}: {source}, Page {page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(formatted)



In [11]:
prompt = ChatPromptTemplate.from_template(
    """You are a helpful telecom policy and tariff assistant.
Answer the question using ONLY the provided context.
If the answer is not clearly in the context, say "I couldn't find that information in the provided telecom documents."
Mention the document name and page number for key claims when available.

Context:
{context}

Question:
{question}

Answer:"""
)

llm = ChatOllama(model=chat_model, temperature=0.2)

rag_chain = (
    {
        "context": retriever | format_docs_with_sources,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [12]:
#ask questions & inspect sources

def ask_telecom_rag(question: str):
    docs = retriever.invoke(question)
    answer = rag_chain.invoke(question)

    print("ANSWER:\n" + answer)
    print("\nSOURCES RETRIEVED:")
    for doc in docs:
        source = Path(doc.metadata.get("source", "unknown")).name
        page = doc.metadata.get("page", 0) + 1
        print(f"  - {source} (Page {page})")


ask_telecom_rag("What are the rules or validity conditions for international roaming packs?")

ANSWER:
According to the provided documents, here are the rules and validity conditions for international roaming packs:

1. International Roaming packs benefits are not applicable while calling local toll free numbers abroad. If a customer tries calling such local toll free number abroad, he/she will be charged according to standard rates applicable in that country. (Source 1: tnc.pdf, Page 6; Source 3: ir-pack-tnc-19042023.pdf, Page 30)
2. For Postpaid customers, the validity of International Roaming Pack benefit works independent of the bill cycle of their postpaid bill plan back in India. (Source 3: ir-pack-tnc-19042023.pdf, Page 31)
3. Postpaid customers can buy a new international roaming pack if they do not have any active international roaming pack. (Source 2: tnc.pdf, Page 6)
4. Postpaid customers will have an option to buy additional High speed data top-up only when they reach international destination and their international roaming pack gets activated. (Source 2: tnc.pdf, P

In [13]:
ask_telecom_rag("What are the terms and conditions regarding prepaid recharge failures or refunds?")


ANSWER:
I couldn't find that information in the provided telecom documents.

The provided documents do not mention the terms and conditions regarding prepaid recharge failures or refunds.

SOURCES RETRIEVED:
  - TRAI_Handbook_2018_Eng.pdf (Page 34)
  - ir-pack-tnc-19042023.pdf (Page 4)
  - TRAI_Handbook_2018_Eng.pdf (Page 16)
  - CR_TTO_28032025.pdf (Page 20)


In [14]:
ask_telecom_rag("How can a telecom consumer register a complaint or escalate an unresolved billing issue?")

ANSWER:
According to the provided context, a telecom consumer can register a complaint or escalate an unresolved billing issue through the following methods:

1. Sending an SMS to the short code 1909 (as mentioned in Source 2: TRAI_Handbook_2018_Eng.pdf, Page 30).
2. Approaching the Complaint Centre of their service provider through the toll-free "Consumer Care Number" (as mentioned in Source 3: TRAI_Handbook_2018_Eng.pdf, Page 17).

Additionally, the complaint must be made within three days of receipt of the unsolicited commercial communication (as mentioned in Source 2: TRAI_Handbook_2018_Eng.pdf, Page 30).

SOURCES RETRIEVED:
  - TRAI_Handbook_2018_Eng.pdf (Page 20)
  - TRAI_Handbook_2018_Eng.pdf (Page 30)
  - TRAI_Handbook_2018_Eng.pdf (Page 17)
  - TRAI_Handbook_2018_Eng.pdf (Page 31)


In [15]:
ask_telecom_rag("What is the reporting requirement for telecom operators when introducing a new tariff plan?")


ANSWER:
According to Source 1: CR_TTO_28032025.pdf, Page 3, the reporting requirement for telecom operators when introducing a new tariff plan is to report to the Authority at least FIVE working days before the proposed date of implementation of any new tariff for telecommunication services under this Order and/or any changes therein, for the approval of the Authority.

Additionally, Source 9: CR_TTO_28032025.pdf, Page 9, states that all service providers shall comply with the reporting requirement in respect of tariffs specified for the first time under this Order and also all subsequent changes, provided that in respect of tariff plans offered by a telecom access provider or Internet service provider to bulk customers, the reporting requirement as defined in sub-clause (i) of clause 2 shall not apply.

However, for bulk customers, the reporting requirement still applies, and the telecom operator must report within SEVEN days after the close of every Quarter, consisting of three month

In [16]:
# out of scope question (hallucination)
ask_telecom_rag("What is the monthly price of JioFiber's 1 Gbps broadband plan in 2026?")


ANSWER:
I couldn't find that information in the provided telecom documents. The provided documents do not mention the monthly price of JioFiber's 1 Gbps broadband plan in 2026.

SOURCES RETRIEVED:
  - CR_TTO_28032025.pdf (Page 26)
  - TRAI_Handbook_2018_Eng.pdf (Page 45)
  - TRAI_Handbook_2018_Eng.pdf (Page 14)
  - TRAI_Handbook_2018_Eng.pdf (Page 14)
